# CEFR Inference Pipeline

Loads the models trained + saved by `cefr_2_methods_reshaped.ipynb` and scores **new learners**.

**Needs, in the same folder:**
- `cefr_common.py`  (the shared model class + scoring code)
- `cefr_models.joblib`  (the saved bundle - produced by the training notebook's last cell)

For each learner you get, per model (`rf`, `gbm`):
`m1`, `m2`, `raw_score`, `bell_score`, `perband_score`, and the predicted `band`.

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import joblib

# importing FrankHallOrdinal binds it into this notebook's namespace so the saved
# pipelines unpickle cleanly; score_dataframe does the whole inference in one call.
from cefr_common import score_dataframe, FrankHallOrdinal

print("cefr_common loaded")

## 1. Load the saved models

In [ ]:
MODEL_PATH = "cefr_models.joblib"
bundle = joblib.load(MODEL_PATH)

print("loaded:", MODEL_PATH)
print("  models         :", list(bundle["models"]))
print("  reshape config :", bundle["reshape_cfg"])
print("  expects these feature columns:")
for c in bundle["feature_cols"]:
    print("    -", c)

## 2. Load the new learners to score   <-- FILL THIS IN

`new_df` must contain the **feature columns listed above** (same names). Optional passthrough
columns `ciid`, `location`, `split` are carried into the output if present.

In [ ]:
# TODO: load the learners you want to score
new_df = None

# e.g. new_df = pd.read_csv("new_learners.csv")
# e.g. new_df = pd.read_excel("new_learners.xlsx")

## 3. Score

In [ ]:
assert new_df is not None, "Load your learners into `new_df` in section 2."

preds = score_dataframe(bundle, new_df)     # keep_cols=("ciid","location","split") by default

print(f"scored {len(preds)} learners x {len(bundle['models'])} models")
with pd.option_context("display.max_rows", 500, "display.max_columns", 60):
    display(preds)

# preds.to_csv("inference_predictions.csv", index=False)   # uncomment to export

## What the columns mean

Per model `k` in `{rf, gbm}`:

| column | meaning |
|---|---|
| `{k}_m1` | P(above A1-A2) - first Frank-Hall sub-model |
| `{k}_m2` | P(above B1) - second Frank-Hall sub-model |
| `{k}_raw_score` | `50*(m1+m2)` - the original 0-100 score (U-shaped) |
| `{k}_bell_score` | reshaped variation **A** (global bell centred ~50) |
| `{k}_perband_score` | reshaped variation **B** (band 0 low, B1 ~50, band 2 high) |
| `{k}_band` | predicted CEFR band (identical across all three scores) |

Pick whichever score column your business uses. The `band` is the same regardless of scoring
variation (the reshapes are monotonic).